# Lab 21-11 · Link Budgets

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import linkbudget as lb

## Part 1 · The wearable ledger

*Predict* the received power at 10 and 30 m for the watch-to-phone link of Example 21-11.1.

In [2]:
def pl_office(d, f_ghz=2.44):
    return 23.9 * np.log10(d) + 30.13 + 24.0 * math.log10(f_ghz)


eirp, body, g_rx = 0 - 5, 5, 0
for d in (10, 20, 30):
    pr = eirp - pl_office(d) - body + g_rx
    print(f"{d:2d} m: received {pr:.2f} dBm, 1M margin {pr + 96.5:.2f} dB, spec margin {pr + 70:.2f} dB")
d = np.linspace(4, 30, 200)
plt.figure(); plt.plot(d, eirp - pl_office(d) - body, label="received")
for s, lab_ in ((-70, "specification"), (-96.5, "1M part"), (-102, "Coded part")):
    plt.axhline(s, ls="--", label=lab_)
plt.xlabel("distance (m)"); plt.ylabel("dBm"); plt.legend(); plt.show()

10 m: received -73.33 dBm, 1M margin 23.17 dB, spec margin -3.33 dB
20 m: received -80.52 dBm, 1M margin 15.98 dB, spec margin -10.52 dB
30 m: received -84.73 dBm, 1M margin 11.77 dB, spec margin -14.73 dB


## Part 2 · Combined margins

*Predict* the outage at the 15.03 dB combined margin for σ = 8 dB.

In [3]:
m = lb.combined_margin_db(8, 0.1)
print(f"combined margin {m:.2f} dB; separate margins add to {lb.lognormal_margin_db(8, 0.1) + lb.rayleigh_margin_db(0.1):.2f} dB")
print(f"simulated outage {100 * lb.mc_outage(m, 8):.2f} %")
ms = np.linspace(0, 40, 81)
plt.figure()
plt.semilogy(ms, [1 - math.exp(-10 ** (-x / 10)) for x in ms], label="Rayleigh")
plt.semilogy(ms, [0.5 * math.erfc(x / (8 * math.sqrt(2))) for x in ms], label="shadowing, 8 dB")
plt.semilogy(ms, [lb.outage_shadow_rayleigh(x, 8) for x in ms], label="both")
plt.ylim(1e-3, 1); plt.xlabel("margin (dB)"); plt.ylabel("outage"); plt.legend(); plt.show()

combined margin 15.03 dB; separate margins add to 20.03 dB
simulated outage 10.00 %


## Part 3 · Area coverage

*Predict* the area coverage for 75 % at the edge (σ = 8 dB, n = 3.5).

In [4]:
print(f"Jakes {100 * lb.edge_to_area_coverage(0.75, 8, 3.5):.2f} %, simulation {100 * lb.mc_area_coverage(0.75, 8, 3.5):.2f} %")

Jakes 89.89 %, simulation 89.91 %


## Part 4 · Repetition

*Predict* the SNR gain of combining 32 copies of a BPSK symbol.

In [5]:
for n in (2, 8, 32):
    print(f"{n:2d} copies: measured {lb.mc_repetition_gain_db(n):.2f} dB, ideal {10 * math.log10(n):.2f} dB")

 2 copies: measured 3.02 dB, ideal 3.01 dB


 8 copies: measured 9.06 dB, ideal 9.03 dB


32 copies: measured 15.03 dB, ideal 15.05 dB


## Part 5 · Satellite elevation

*Predict* the uplink CNR overhead and at 20° for the LEO-600 handheld link.

In [6]:
els = np.arange(10, 91, 5)
cnr = []
for el in els:
    d = lb.slant_range_m(600e3, el)
    cnr.append(lb.cn0_dbhz(-7, lb.fspl_db(2e9, d), 1.1, 5.3) - 10 * math.log10(360e3))
for el in (90, 30, 20):
    d = lb.slant_range_m(600e3, el)
    print(f"{el:2d} deg: {d/1e3:.0f} km, path loss {lb.fspl_db(2e9, d):.2f} dB, "
          f"CNR {lb.cn0_dbhz(-7, lb.fspl_db(2e9, d), 1.1, 5.3) - 10 * math.log10(360e3):.2f} dB")
plt.figure(); plt.plot(els, cnr); plt.axhline(0, ls=":"); plt.xlabel("elevation (deg)"); plt.ylabel("uplink CNR in 360 kHz (dB)"); plt.show()

90 deg: 600 km, path loss 154.03 dB, CNR 7.81 dB
30 deg: 1075 km, path loss 159.10 dB, CNR 2.74 dB
20 deg: 1392 km, path loss 161.34 dB, CNR 0.49 dB


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.